# canvas-host: simple WebGL2 example

This example shows the minimal worker-side setup:

1. attach an `OffscreenCanvas` and transfer it to the kernel worker
2. create a WebGL2 context
3. start the normalized input event queue
4. poll pointer and keyboard events in a render loop
5. stop the loop and release the canvas

The shader draws a glow under the pointer. Clicking the canvas or holding a key
changes its color. All cells are C++ and use `<canvas_host/canvas_host.hpp>`.


In [ ]:
#include <canvas_host/canvas_host.hpp>

#include <iostream>

canvas_host::WorkerConfig config;
config.width = 640;
config.height = 400;
config.status_text = "canvas-host / simple WebGL2";

auto canvas = canvas_host::attach_worker(config);

std::cout << "canvas id: " << canvas.id() << "\n";


## Create the context and start rendering

Run this cell after the canvas output has rendered. The browser transfers the
canvas to the worker asynchronously. `require_ready()` does not wait; it throws
a clear error until the transfer completes, so rerun this cell if needed.


In [ ]:
#include <canvas_host/canvas_host.hpp>

#include <GLES3/gl3.h>
#include <emscripten.h>
#include <emscripten/eventloop.h>

#include <iostream>
#include <string>

canvas.require_ready();

int g_webgl_ctx = canvas_host::create_webgl2_context(canvas);
int g_pixel_width = canvas.pixel_width();
int g_pixel_height = canvas.pixel_height();

canvas.start_events();

static GLuint g_program = 0;
static GLuint g_vao = 0;
static GLint g_pointer_location = -1;
static GLint g_pressed_location = -1;
static int g_interval = 0;
static float g_pointer_x = 0.5f;
static float g_pointer_y = 0.5f;
static float g_pressed = 0.0f;

static constexpr const char* g_vertex_shader = R"GLSL(#version 300 es
out vec2 v_uv;
void main()
{
    vec2 p = vec2(float((gl_VertexID << 1) & 2), float(gl_VertexID & 2));
    v_uv = p;
    gl_Position = vec4(p * 2.0 - 1.0, 0.0, 1.0);
}
)GLSL";

static constexpr const char* g_fragment_shader = R"GLSL(#version 300 es
precision highp float;
in vec2 v_uv;
uniform vec2 u_pointer;
uniform float u_pressed;
out vec4 frag_color;
void main()
{
    float glow = 1.0 - smoothstep(0.0, 0.35, distance(v_uv, u_pointer));
    vec3 base = mix(vec3(0.08, 0.25, 0.55), vec3(0.85, 0.35, 0.10), u_pressed);
    frag_color = vec4(base + glow, 1.0);
}
)GLSL";

static GLuint compile_shader(GLenum type, const char* source)
{
    GLuint shader = glCreateShader(type);
    glShaderSource(shader, 1, &source, nullptr);
    glCompileShader(shader);

    GLint ok = GL_FALSE;
    glGetShaderiv(shader, GL_COMPILE_STATUS, &ok);
    if (ok == GL_FALSE) {
        char log[4096] = {};
        glGetShaderInfoLog(shader, static_cast<GLsizei>(sizeof(log)), nullptr, log);
        std::cout << "Shader compile error: " << log << "\n";
        glDeleteShader(shader);
        return 0;
    }
    return shader;
}

static GLuint link_program(GLuint vertex_shader, GLuint fragment_shader)
{
    GLuint program = glCreateProgram();
    glAttachShader(program, vertex_shader);
    glAttachShader(program, fragment_shader);
    glLinkProgram(program);

    GLint ok = GL_FALSE;
    glGetProgramiv(program, GL_LINK_STATUS, &ok);
    if (ok == GL_FALSE) {
        char log[4096] = {};
        glGetProgramInfoLog(program, static_cast<GLsizei>(sizeof(log)), nullptr, log);
        std::cout << "Program link error: " << log << "\n";
        glDeleteProgram(program);
        return 0;
    }
    return program;
}

static void render_frame()
{
    glViewport(0, 0, g_pixel_width, g_pixel_height);
    glClearColor(0.02f, 0.03f, 0.06f, 1.0f);
    glClear(GL_COLOR_BUFFER_BIT);

    glUseProgram(g_program);
    glUniform2f(g_pointer_location, g_pointer_x, g_pointer_y);
    glUniform1f(g_pressed_location, g_pressed);
    glBindVertexArray(g_vao);
    glDrawArrays(GL_TRIANGLES, 0, 3);
    glBindVertexArray(0);
}

static void on_tick(void*)
{
    canvas_host::nl::json event;
    while (canvas.poll_event(event)) {
        const std::string type = event.value("type", std::string());
        if (type == "pointermove" || type == "pointerdown" || type == "pointerup") {
            const double css_width = event.value("cssWidth", 1.0);
            const double css_height = event.value("cssHeight", 1.0);
            g_pointer_x = static_cast<float>(event.value("x", 0.0) / css_width);
            g_pointer_y = 1.0f - static_cast<float>(event.value("y", 0.0) / css_height);
        }
        if (type == "pointerdown" || type == "keydown") {
            g_pressed = 1.0f;
        } else if (type == "pointerup" || type == "pointercancel" || type == "keyup") {
            g_pressed = 0.0f;
        }
    }
    render_frame();
}

void webgl_example_stop()
{
    if (g_interval != 0) {
        emscripten_clear_interval(g_interval);
        g_interval = 0;
    }
    if (g_program != 0) {
        glDeleteProgram(g_program);
        g_program = 0;
    }
    if (g_vao != 0) {
        glDeleteVertexArrays(1, &g_vao);
        g_vao = 0;
    }
    if (g_webgl_ctx != 0) {
        canvas.destroy_webgl2_context(g_webgl_ctx);
        g_webgl_ctx = 0;
    }
}

if (g_webgl_ctx == 0) {
    std::cout << "WebGL2 context creation failed\n";
} else {
    const GLuint vertex_shader = compile_shader(GL_VERTEX_SHADER, g_vertex_shader);
    const GLuint fragment_shader = compile_shader(GL_FRAGMENT_SHADER, g_fragment_shader);
    if (vertex_shader != 0 && fragment_shader != 0) {
        g_program = link_program(vertex_shader, fragment_shader);
    }
    glDeleteShader(vertex_shader);
    glDeleteShader(fragment_shader);

    if (g_program == 0) {
        std::cout << "Shader setup failed\n";
    } else {
        g_pointer_location = glGetUniformLocation(g_program, "u_pointer");
        g_pressed_location = glGetUniformLocation(g_program, "u_pressed");
        glGenVertexArrays(1, &g_vao);
        glBindVertexArray(g_vao);
        render_frame();
        glBindVertexArray(0);
        g_interval = emscripten_set_interval(on_tick, 16, nullptr);
        std::cout << "WebGL2 " << g_pixel_width << "x" << g_pixel_height
                  << " px; move the pointer, click, or type. Events are polled "
                     "with canvas.poll_event().\n";
    }
}


## Stop

Stop the render loop, destroy the WebGL2 context, and close the canvas-host
channels.


In [ ]:
#include <iostream>

void webgl_example_stop();

webgl_example_stop();
canvas.stop();
std::cout << "canvas stopped\n";
